# La Liga Market Value Analysis - Data Cleaning

`01_data_cleaning.ipynb` this notebook loads and cleans the two data sources I used for the La Liga market value analysis.


| # | Section | Output |
|---|---------|--------|
| 1 | Data Loading | Raw DataFrames inspected |
| 2 | Filter Transfermarkt for La Liga | ES1-only subsets, season labels |
| 3 | Data Quality Check | Null/duplicate reports, distributions |
| 4 | Merge | `laliga_merged.csv` |


---
## 1. Data Loading

This project uses two data sources:

**Understat** (`understat_laliga_2014_2024.csv`) scraped from understat.com using the site's internal API. Each row is one player's stats for one La Liga season, with metrics like xG, xA, goals, assists, and minutes played.

**Transfermarkt** (files in `transfermarkt_data/`) downloaded from Kaggle (author `davidcariboo/player-scores`, CC0 licence). The three files used here are:
- `players.csv` - basic player info (position, nationality, date of birth, market value)
- `player_valuations.csv` - a history of market value snapshots per player and date
- `competitions.csv` - used to find the La Liga competition code (`ES1`)


In [1]:
# importar librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# ruta de los archivos de transfermarkt
ruta_tm = "transfermarkt_data/"

print("librerias ok")

librerias ok


### 1.1 Understat Player Statistics


In [2]:
# cargar el dataset de understat
understat = pd.read_csv("understat_laliga_2014_2024.csv")

print(f"filas y columnas: {understat.shape}")
print(f"\nColumnas ({len(understat.columns)}):")
print(list(understat.columns))


filas y columnas: (6192, 15)

Columnas (15):
['player_id', 'player_name', 'games', 'time', 'goals', 'xG', 'assists', 'xA', 'shots', 'key_passes', 'position', 'team_title', 'npg', 'npxG', 'season']


In [3]:
understat.head()


,player_id,player_name,games,time,goals,xG,assists,xA,shots,key_passes,position,team_title,npg,npxG,season
0,2371,Cristiano Ronaldo,35,3103,48,39.308746,16,13.600629,225,76,F M,Real Madrid,38,30.389681,2014/15
1,2097,Lionel Messi,38,3374,43,35.891770,18,17.611949,187,95,F S,Barcelona,38,31.432102,2014/15
2,2099,Neymar,33,2573,22,22.721615,7,8.261539,95,52,F S,Barcelona,21,21.978335,2014/15
3,2270,Antoine Griezmann,37,2490,22,14.709659,1,2.607721,78,23,F M S,Atletico Madrid,22,14.709659,2014/15
4,1125,Carlos Bacca,37,2581,20,19.279248,6,4.728216,69,31,F S,Sevilla,14,13.333664,2014/15


### 1.2 Transfermarkt - `players.csv`


In [4]:
# cargar metadatos estaticos de jugadores
tm_players = pd.read_csv(ruta_tm + "players.csv")

print(f"filas y columnas: {tm_players.shape}")
print(f"\nColumnas ({len(tm_players.columns)}):")
print(list(tm_players.columns))


filas y columnas: (34320, 23)

Columnas (23):
['player_id', 'first_name', 'last_name', 'name', 'last_season', 'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth', 'country_of_citizenship', 'date_of_birth', 'sub_position', 'position', 'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name', 'image_url', 'url', 'current_club_domestic_competition_id', 'current_club_name', 'market_value_in_eur', 'highest_market_value_in_eur']


In [5]:
tm_players.head()


,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,foot,height_in_cm,contract_expiration_date,agent_name,image_url,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
0,10,Miroslav,Klose,Miroslav Klose,2015,398,miroslav-klose,Poland,Opole,Germany,...,right,184.0,NaN,ASBW Sport Marketing,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/miroslav-klose...,IT1,Società Sportiva Lazio S.p.A.,1000000.0,30000000.0
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,...,left,190.0,NaN,Neubauer 13 GmbH,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/roman-weidenfe...,L1,Borussia Dortmund,750000.0,8000000.0
2,65,Dimitar,Berbatov,Dimitar Berbatov,2015,1091,dimitar-berbatov,Bulgaria,Blagoevgrad,Bulgaria,...,NaN,NaN,NaN,CSKA-AS-23 Ltd.,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/dimitar-berbat...,GR1,Panthessalonikios Athlitikos Omilos Konstantin...,1000000.0,34500000.0
3,77,NaN,Lúcio,Lúcio,2012,506,lucio,Brazil,Brasília,Brazil,...,NaN,NaN,NaN,NaN,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/lucio/profil/s...,IT1,Juventus Football Club,200000.0,24500000.0
4,80,Tom,Starke,Tom Starke,2017,27,tom-starke,East Germany (GDR),Freital,Germany,...,right,194.0,NaN,IFM,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/tom-starke/pro...,L1,FC Bayern München,100000.0,3000000.0


### 1.3 Transfermarkt - `player_valuations.csv`


In [6]:
# cargar historial de valoraciones de mercado
tm_val = pd.read_csv(ruta_tm + "player_valuations.csv")

print(f"filas y columnas: {tm_val.shape}")
print(f"\nColumnas ({len(tm_val.columns)}):")
print(list(tm_val.columns))


filas y columnas: (435443, 6)

Columnas (6):
['player_id', 'date', 'market_value_in_eur', 'current_club_name', 'current_club_id', 'player_club_domestic_competition_id']


In [7]:
tm_val.head()


,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id
0,405973,2000-01-20,150000,Unknown,3057.0,BE1
1,342216,2001-07-20,100000,Unknown,1241.0,SC1
2,3132,2003-12-09,400000,Dynamo Kyiv,126.0,TR1
3,6893,2003-12-15,900000,Galatasaray,984.0,GB1
4,12359,2004-03-11,250000,RC Lens B,8152.0,NaN


### 1.4 Transfermarkt - `competitions.csv`


In [8]:
# cargar catalogo de competiciones y mostrar la fila de la liga
tm_comp = pd.read_csv(ruta_tm + "competitions.csv")

print(f"filas y columnas: {tm_comp.shape}")
print(f"\nFila ES1 (La Liga):")
print(tm_comp[tm_comp['competition_id'] == 'ES1'].to_string())


filas y columnas: (44, 11)

Fila ES1 (La Liga):
   competition_id competition_code    name    sub_type             type  country_id country_name domestic_league_code confederation                                                               url  is_major_national_league
14            ES1           laliga  laliga  first_tier  domestic_league         157        Spain                  ES1        europa  https://www.transfermarkt.co.uk/laliga/startseite/wettbewerb/ES1                      True


---
## 2. Filter Transfermarkt for La Liga

The Transfermarkt files cover many different leagues. Here I keep only La Liga (competition_id `ES1`) entries and limit the years to 2014–2024 to match the Understat data.

### Season convention
A football season spans two calendar years (e.g. 2014/15 runs from August 2014 to May 2015).
I assigned each valuation date to a season using this rule

| Month of valuation date | Season start year |
|-------------------------|-------------------|
| August - December | Same year |
| January - July | Year before |

As an example a valuation from March 2015 belongs to season 2014/15, while one from September 2015 belongs to 2015/16.


In [9]:
# jugadores con club actual en la liga
tm_players_es1 = tm_players[
    tm_players['current_club_domestic_competition_id'] == 'ES1'
].copy()

print(f"players.csv - total:          {len(tm_players):,}")
print(f"players.csv - ES1 only:       {len(tm_players_es1):,}")

print(f"jugadores en La Liga: {len(tm_players_es1)}")

players.csv - total:          34,320
players.csv - ES1 only:       2,331
jugadores en La Liga: 2331


In [10]:
# valoraciones correspondientes a la liga
tm_val_es1 = tm_val[
    tm_val['player_club_domestic_competition_id'] == 'ES1'
].copy()

print(f"player_valuations.csv - total:     {len(tm_val):,}")
print(f"player_valuations.csv - ES1 only:  {len(tm_val_es1):,}")


player_valuations.csv - total:     435,443
player_valuations.csv - ES1 only:  33,284


In [11]:
# filtrado 2014-2024 y asignar etiqueta de temporada

# convertir columna de fecha a datetime
tm_val_es1['date'] = pd.to_datetime(tm_val_es1['date'])

# extraer año y filtrar rango
tm_val_es1['year'] = tm_val_es1['date'].dt.year
mask_years = tm_val_es1['year'].between(2014, 2024)
tm_val_es1 = tm_val_es1[mask_years].copy()

print(f"player_valuations.csv - ES1, 2014-2024: {len(tm_val_es1):,}")
print(f"Rango de fechas: {tm_val_es1['date'].min().date()}  a  {tm_val_es1['date'].max().date()}")

print(f"valoraciones en rango 2014-2024: {len(tm_val_es1)}")

player_valuations.csv - ES1, 2014-2024: 24,426
Rango de fechas: 2014-01-02  a  2024-12-03
valoraciones en rango 2014-2024: 24426


In [12]:
# calcular el año de inicio de temporada segun el mes de la valoracion
def asignar_temporada(fila):
    anio = fila['year']
    mes = fila['date'].month
    if mes >= 8:
        return anio
    else:
        return anio - 1

tm_val_es1['season_start'] = tm_val_es1.apply(asignar_temporada, axis=1)

# crear etiqueta de temporada en formato "yyyy/yy"
def formato_temporada(y):
    fin = str(y + 1)[-2:]
    return f"{y}/{fin}"

tm_val_es1['season'] = tm_val_es1['season_start'].apply(formato_temporada)

# excluir valoraciones de 2013/14 que aparecen por fechas de enero-julio 2014
tm_val_es1 = tm_val_es1[tm_val_es1['season_start'] >= 2014].copy()

print(f"Filas tras excluir temporada 2013/14: {len(tm_val_es1):,}")
print(f"Temporadas únicas: {sorted(tm_val_es1['season'].unique())}")


Filas tras excluir temporada 2013/14: 23,382
Temporadas únicas: ['2014/15', '2015/16', '2016/17', '2017/18', '2018/19', '2019/20', '2020/21', '2021/22', '2022/23', '2023/24', '2024/25']


In [13]:
# quedarse con la ultima valoracion por jugador y temporada
tm_sorted = tm_val_es1.sort_values('date')
tm_val_season = tm_sorted.drop_duplicates(
    subset=['player_id', 'season'], keep='last'
).reset_index(drop=True)
print(f"snapshots unicos por jugador y temporada: {len(tm_val_season)}")

print(f"Total de observaciones finales (player_id × season): {len(tm_val_season):,}")
print(f"Cantidad de jugadores distintos con al menos una valoracion: {tm_val_season['player_id'].nunique():,}")
tm_val_season[['player_id', 'season', 'date', 'market_value_in_eur', 'current_club_name']].head(8)


snapshots unicos por jugador y temporada: 10995
Total de observaciones finales (player_id × season): 10,995
Cantidad de jugadores distintos con al menos una valoracion: 2,230


,player_id,season,date,market_value_in_eur,current_club_name
0,120582,2014/15,2014-08-04,100000,Atlético Malagueño
1,303248,2014/15,2014-08-04,100000,Real Betis B
2,44700,2014/15,2014-08-04,1200000,Real Betis Balompié
3,16124,2014/15,2014-08-05,400000,CD Tenerife
4,191754,2014/15,2014-08-10,200000,Recreativo Huelva
5,288453,2014/15,2014-08-10,100000,Recreativo Huelva B
6,221984,2014/15,2014-08-10,200000,AD Alcorcón
7,31067,2014/15,2014-08-11,7500000,SSC Napoli


---
## 3. Data Quality Check

I check missing values, duplicate keys, and the distributions of the main variables before merging **player position** (Understat) and **market value** (Transfermarkt).


### 3.1 Missing values


In [14]:
# nulos en understat
print("understat - valores faltantes:")
null_us = understat.isnull().sum()
if null_us.sum() == 0:
    print("sin valores faltantes")
else:
    print(null_us[null_us > 0].to_string())
    print(f"  Total cells faltantes: {null_us.sum()}")


understat - valores faltantes:
sin valores faltantes


In [15]:
# nulos en valoraciones transfermarkt
print("transfermarkt (ES1, 2014-2024) - valores faltantes:")
null_tm = tm_val_season.isnull().sum()
nulos_tm = null_tm[null_tm > 0]
if len(nulos_tm) == 0:
    print("sin valores faltantes")
else:
    pct = (nulos_tm / len(tm_val_season) * 100).round(1)
    summary = pd.DataFrame({'missing': nulos_tm, 'pct (%)': pct})
    print(summary.to_string())


transfermarkt (ES1, 2014-2024) - valores faltantes:
sin valores faltantes


### 3.2 Duplicate keys


In [16]:
# duplicados en understat (player_id, season)
dup_us = understat.duplicated(subset=['player_id', 'season']).sum()

# duplicados en transfermarkt (player_id, season)
dup_tm = tm_val_season.duplicated(subset=['player_id', 'season']).sum()

print(f"understat - filas duplicadas (player_id, season): {dup_us}")
print(f"transfermarkt - filas duplicadas (player_id, season): {dup_tm}")
print()
if dup_us == 0 and dup_tm == 0:
    print("sin duplicados en ninguno de los dos datasets, ok para hacer el merge")


understat - filas duplicadas (player_id, season): 0
transfermarkt - filas duplicadas (player_id, season): 0

sin duplicados en ninguno de los dos datasets, ok para hacer el merge


---
## 4. Merge Understat and Transfermarkt

I merged the two datasets by matching player name and season.
First I cleaned both name columns - removed accents, lowercased, removed punctuation -
so that 'Lionel Messi' and 'lionel messi' would match. Then I did an exact join
on (cleaned_name, season).

Players from Understat with no match in Transfermarkt are excluded from the analysis
because they have no market value data.

In [17]:
# añadir nombre del jugador a la tabla de valoraciones
# player_valuations solo tiene player_id; el nombre viene de players.csv
tm_names = tm_players[['player_id', 'name']].rename(columns={'name': 'tm_name'})

tm_val_named = tm_val_season.merge(tm_names, on='player_id', how='left')

sin_nombre = tm_val_named['tm_name'].isna().sum()
print(f"Valoraciones (ES1) con nombre del jugador: {len(tm_val_named):,}")
print(f"  player_id sin match en players.csv:  {sin_nombre:,}")


Valoraciones (ES1) con nombre del jugador: 10,995
  player_id sin match en players.csv:  1


In [18]:
# normalizar el nombre para el merge (minusculas, sin espacios extra)
def normalize_name(name):
    if pd.isna(name) or str(name).strip() == '':
        return ''
    return str(name).lower().strip()
# aplicar normalizacion de nombres en los datasets
understat['name_norm']    = understat['player_name'].apply(normalize_name)
tm_val_named['name_norm'] = tm_val_named['tm_name'].apply(normalize_name)
# verificacion con ejemplos
sample = (understat[['player_name', 'name_norm']]
          .drop_duplicates()
          .sample(10, random_state=42))
print("ejemplos de verificacion:")
print(sample.to_string(index=False))


ejemplos de verificacion:
      player_name         name_norm
     Rubén Duarte      rubén duarte
     Luis Martins      luis martins
   Yannis Salibur    yannis salibur
   Dani Rodríguez    dani rodríguez
    Jaime Gavilán     jaime gavilán
            Luque             luque
Eduard Campabadal eduard campabadal
   Julián Álvarez    julián álvarez
  Matija Nastasic   matija nastasic
  Largie Ramazani   largie ramazani


In [19]:
# Merge
# join exacto por (name_norm, season)

# columnas de transfermarkt para incorporar
cols_tm = ['name_norm', 'season', 'player_id', 'tm_name',
           'market_value_in_eur', 'date', 'current_club_name']

exact_merged = understat.merge(
    tm_val_named[cols_tm].rename(columns={
        'player_id':    'tm_player_id',
        'date':         'valuation_date',
        'current_club_name': 'tm_club',
    }),
    on=['name_norm', 'season'],
    how='inner'
)

# filas de understat sin match exacto
matched_keys = set(zip(exact_merged['name_norm'], exact_merged['season']))
unmatched_us = understat.copy()
mask = []
for _, r in unmatched_us.iterrows():
    mask.append((r['name_norm'], r['season']) in matched_keys)
mask_invertido = []
for m in mask:
    mask_invertido.append(not m)
unmatched_us = unmatched_us[mask_invertido].copy()
print(f"understat total:          {len(understat):,}")
print(f"  matcheados (exacto):    {len(exact_merged):,}")
print(f"  sin match exacto:       {len(unmatched_us):,}")


understat total:          6,192
  matcheados (exacto):    3,388
  sin match exacto:       2,854


In [20]:
# resultado del merge exacto
merged = exact_merged.copy()
merged['match_type'] = 'exact'

print(f"join exacto: {len(merged)} filas matcheadas")
print(f"understat sin match: {len(understat) - len(merged)}")
print(f"match rate: {len(merged)/len(understat)*100:.1f}%")

join exacto: 3388 filas matcheadas
understat sin match: 2804


### Save merged dataset


In [21]:
# guardar el dataset merged como laliga_merged.csv
archivo_merged = "laliga_merged.csv"
merged.to_csv(archivo_merged, index=False, encoding='utf-8')

print(f"guardado: {archivo_merged}")
print(f"filas y columnas: {merged.shape}")
print(f"columnas: {list(merged.columns)}")


guardado: laliga_merged.csv
filas y columnas: (3388, 22)
columnas: ['player_id', 'player_name', 'games', 'time', 'goals', 'xG', 'assists', 'xA', 'shots', 'key_passes', 'position', 'team_title', 'npg', 'npxG', 'season', 'name_norm', 'tm_player_id', 'tm_name', 'market_value_in_eur', 'valuation_date', 'tm_club', 'match_type']


In [22]:
# vista previa del dataset fusionado
merged.head()


,player_id,player_name,games,time,goals,xG,assists,xA,shots,key_passes,...,npg,npxG,season,name_norm,tm_player_id,tm_name,market_value_in_eur,valuation_date,tm_club,match_type
0,2270,Antoine Griezmann,37,2490,22,14.709659,1,2.607721,78,23,...,22,14.709659,2014/15,antoine griezmann,125781,Antoine Griezmann,45000000,2015-01-07,Atlético de Madrid,exact
1,1125,Carlos Bacca,37,2581,20,19.279248,6,4.728216,69,31,...,14,13.333664,2014/15,carlos bacca,119235,Carlos Bacca,25000000,2015-07-01,Sevilla FC,exact
2,2098,Luis Suárez,27,2182,16,14.542304,14,13.520651,75,60,...,16,14.542304,2014/15,luis suárez,44352,Luis Suárez,80000000,2015-07-01,FC Barcelona,exact
3,2370,Karim Benzema,29,2325,15,12.412423,10,7.889433,69,45,...,15,12.412423,2014/15,karim benzema,18922,Karim Benzema,50000000,2015-07-01,Real Madrid,exact
4,4060,Sergio García,35,3030,14,10.270765,9,8.130714,80,52,...,12,8.784336,2014/15,sergio garcía,7922,Sergio García,6000000,2015-07-01,Al-Rayyan SC,exact


After the merge I ended up with fewer rows than the original Understat dataset. Players without a match in Transfermarkt have no market value, so they cannot be used in the model. The next step is building the per-90 variables and applying the log transformation to market value.

---
## 5. New Variables and Filters

This section applies filters and creates the variables needed for the regression. I track row counts at each step.

### Filtering decisions

| Filter | Threshold | Why |
|--------|-----------|-----|
| Minimum market value | ≥ €500,000 | Values below this are usually placeholder entries on Transfermarkt, not real market valuations. |
| Minimum minutes played | ≥ 450 min (~ 5 full games) | Per-90 stats are meaningless for players who only played a few minutes. |
| Goalkeepers removed | position starts with `GK` | Comparing goalkeepers with outfield players is not meaningful. GKs rarely generate xG or xA, and their performance is driven by other metrics, so including them would add noise to the analysis.|


In [23]:
# cargar el dataset merged
df = pd.read_csv("laliga_merged.csv")

# convertir columnas numericas que llegaron como string de understat
num_cols = ['games', 'time', 'goals', 'xG', 'assists', 'xA',
            'shots', 'key_passes', 'npg', 'npxG', 'market_value_in_eur']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"laliga_merged.csv cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")

print(df.dtypes)
df.head()

laliga_merged.csv cargado: 3,388 filas × 22 columnas
player_id                int64
player_name                str
games                    int64
time                     int64
goals                    int64
xG                     float64
assists                  int64
xA                     float64
shots                    int64
key_passes               int64
position                   str
team_title                 str
npg                      int64
npxG                   float64
season                     str
name_norm                  str
tm_player_id             int64
tm_name                    str
market_value_in_eur      int64
valuation_date             str
tm_club                    str
match_type                 str
dtype: object


,player_id,player_name,games,time,goals,xG,assists,xA,shots,key_passes,...,npg,npxG,season,name_norm,tm_player_id,tm_name,market_value_in_eur,valuation_date,tm_club,match_type
0,2270,Antoine Griezmann,37,2490,22,14.709659,1,2.607721,78,23,...,22,14.709659,2014/15,antoine griezmann,125781,Antoine Griezmann,45000000,2015-01-07,Atlético de Madrid,exact
1,1125,Carlos Bacca,37,2581,20,19.279248,6,4.728216,69,31,...,14,13.333664,2014/15,carlos bacca,119235,Carlos Bacca,25000000,2015-07-01,Sevilla FC,exact
2,2098,Luis Suárez,27,2182,16,14.542304,14,13.520651,75,60,...,16,14.542304,2014/15,luis suárez,44352,Luis Suárez,80000000,2015-07-01,FC Barcelona,exact
3,2370,Karim Benzema,29,2325,15,12.412423,10,7.889433,69,45,...,15,12.412423,2014/15,karim benzema,18922,Karim Benzema,50000000,2015-07-01,Real Madrid,exact
4,4060,Sergio García,35,3030,14,10.270765,9,8.130714,80,52,...,12,8.784336,2014/15,sergio garcía,7922,Sergio García,6000000,2015-07-01,Al-Rayyan SC,exact


### 5.1 Filter by minimum market value (≥ €500,000)


In [24]:
# valor de mercado minimo para el analisis €500.000
n_before = len(df)
df = df[df['market_value_in_eur'] >= 500_000].copy()
n_after = len(df)

print(f"Antes del filtro de valor minimo:  {n_before:,}")
print(f"Eliminados (< €500,000):           {n_before - n_after:,}  "
      f"({(n_before - n_after)/n_before*100:.1f}%)")
print(f"Después del filtro:                {n_after:,}")


Antes del filtro de valor minimo:  3,388
Eliminados (< €500,000):           243  (7.2%)
Después del filtro:                3,145


### 5.2 Filter by minimum minutes played (≥ 450 min)


In [25]:
# minutos minimos jugados para el analisis (450 min)
n_before = len(df)
df = df[df['time'] >= 450].copy()
n_after = len(df)

print(f"Antes del filtro de minutos:       {n_before:,}")
print(f"Eliminados (< 450 min):            {n_before - n_after:,}  "
      f"({(n_before - n_after)/n_before*100:.1f}%)")
print(f"Despues del filtro:                {n_after:,}")


Antes del filtro de minutos:       3,145
Eliminados (< 450 min):            487  (15.5%)
Despues del filtro:                2,658


### 5.3 Filter by only outfield players (remove GK)


In [26]:
# eliminar arqueros (GK)
# en understat los arqueros aparecen como 'gk' o 'gk s'
mask_gk = df['position'].str.strip().str.startswith('GK', na=False)

n_before = len(df)
df = df[~mask_gk].copy()
n_after = len(df)

print(f"Antes de eliminar arqueros:        {n_before:,}")
print(f"Arqueros eliminados:               {n_before - n_after:,}  "
      f"({(n_before - n_after)/n_before*100:.1f}%)")
print(f"Despues del filtro:                {n_after:,}")


Antes de eliminar arqueros:        2,658
Arqueros eliminados:               210  (7.9%)
Despues del filtro:                2,448


### 5.4 Per-90-minute metrics

Dividing by minutes played (in 90-minute units) makes possible to compare players fairly, even if one played 500 minutes and another played 3,000.


In [27]:
# metricas per-90 minutos
# denominador: numero de 90-minutos jugados por el jugador en la temporada
df['p90']           = df['time'] / 90.0

df['xG_per90']      = df['xG']      / df['p90']
df['xA_per90']      = df['xA']      / df['p90']
df['goals_per90']   = df['goals']   / df['p90']
df['assists_per90'] = df['assists'] / df['p90']
df['npxG_per90']    = df['npxG']    / df['p90']
df['shots_per90']   = df['shots']   / df['p90']

# verificar que no hay infinitos ni nan (no deberia haber despues del filtro de 450 min)
per90_cols = ['xG_per90','xA_per90','goals_per90','assists_per90','npxG_per90','shots_per90']
print("Per-90 metrics - null/inf check:")
for c in per90_cols:
    n_null = df[c].isna().sum()
    n_inf  = np.isinf(df[c]).sum()
    print(f"  {c:<18}  nulls={n_null}  inf={n_inf}")

print("\nPer-90 metrics:")
print(df[per90_cols].describe().round(3).to_string())


Per-90 metrics - null/inf check:
  xG_per90            nulls=0  inf=0
  xA_per90            nulls=0  inf=0
  goals_per90         nulls=0  inf=0
  assists_per90       nulls=0  inf=0
  npxG_per90          nulls=0  inf=0
  shots_per90         nulls=0  inf=0

Per-90 metrics:


       xG_per90  xA_per90  goals_per90  assists_per90  npxG_per90  shots_per90
count  2448.000  2448.000     2448.000       2448.000    2448.000     2448.000
mean      0.137     0.095        0.127          0.090       0.127        1.217
std       0.154     0.084        0.166          0.102       0.141        0.921
min       0.000     0.000        0.000          0.000       0.000        0.000
25%       0.032     0.029        0.000          0.000       0.032        0.481
50%       0.077     0.074        0.069          0.062       0.074        0.956
75%       0.191     0.141        0.178          0.143       0.171        1.796
max       1.460     0.559        1.167          0.833       1.460        5.569


### 5.5 Log transformation of market value

Market value ranges from €500k to over €200M - a very skewed distribution where a few
superstar values pull the average way up. I plotted the raw distribution and it was
clearly not symmetric. Taking the natural log makes it much more symmetric and easier
to work with in a regression. This is standard practice when a variable spans several
orders of magnitude.

In [28]:
# logaritmo natural del valor de mercado
df['log_market_value'] = np.log(df['market_value_in_eur'])

print(f"log_market_value  -  min: {df['log_market_value'].min():.3f}"
      f"  |  max: {df['log_market_value'].max():.3f}"
      f"  |  mean: {df['log_market_value'].mean():.3f}"
      f"  |  std: {df['log_market_value'].std():.3f}")


log_market_value  -  min: 13.122  |  max: 19.008  |  mean: 15.411  |  std: 1.174


### 5.6 Position simplification and dummy variables

Understat uses compound codes (e.g. `F M S`, `D S`). I extract the **primary position** from the
first token and collapse into four categories. **Defender** is the reference category in all models.


In [29]:
# simplificar posicion al token principal
mapa_pos = {'D': 'Defender', 'M': 'Midfielder', 'F': 'Forward', 'S': 'Forward'}

# devuelve la posicion simplificada segun el primer caracter del codigo
def simplify_position(pos):
    if pd.isna(pos) or str(pos).strip() == '':
        return 'Unknown'
    first_token = str(pos).strip().split()[0]
    return mapa_pos.get(first_token, 'Unknown')

df['position_simple'] = df['position'].apply(simplify_position)

print("Distribución de posición simplificada (tras todos los filtros):")
vc = df['position_simple'].value_counts()
for pos, cnt in vc.items():
    print(f"  {pos:<12} {cnt:,}  ({cnt/len(df)*100:.1f}%)")


Distribución de posición simplificada (tras todos los filtros):
  Defender     1,051  (42.9%)
  Forward      761  (31.1%)
  Midfielder   636  (26.0%)


In [30]:
# dummies de posicion (referencia: defender)
pos_dummies = pd.get_dummies(df['position_simple'], prefix='pos', dtype=int)

# quitar la categoria base para no tener variables redundantes
if 'pos_Defender' in pos_dummies.columns:
    pos_dummies = pos_dummies.drop(columns=['pos_Defender'])

df = pd.concat([df, pos_dummies], axis=1)

dummy_cols = []
for c in df.columns:
    if c.startswith('pos_'):
        dummy_cols.append(c)
print(f"Dummies creados (Defender es la referencia omitida):")
for col in dummy_cols:
    print(f"  {col}: {df[col].sum():,} ones")


Dummies creados (Defender es la referencia omitida):
  pos_Forward: 761 ones
  pos_Midfielder: 636 ones


### 5.7 Season year (numeric)

Extract the starting year of each season as an integer (e.g. `"2014/15"` → `2014`) for use as a
time-trend control variable in regression.


In [31]:
# extraer el año de inicio de la temporada del formato "yyyy/yy"
partes = df['season'].str[:4]
df['season_year'] = partes.astype(int)

print("Distribucion por temporadas (season_year):")
sy = df['season_year'].value_counts().sort_index()
for yr, cnt in sy.items():
    print(f"  {yr} ({yr}/{str(yr+1)[-2:]}):  {cnt:,} player-season records")


Distribucion por temporadas (season_year):
  2014 (2014/15):  225 player-season records
  2015 (2015/16):  205 player-season records
  2016 (2016/17):  245 player-season records
  2017 (2017/18):  258 player-season records
  2018 (2018/19):  159 player-season records
  2019 (2019/20):  265 player-season records
  2020 (2020/21):  307 player-season records
  2021 (2021/22):  322 player-season records
  2022 (2022/23):  106 player-season records
  2023 (2023/24):  283 player-season records
  2024 (2024/25):  73 player-season records


---
## 6. Cleaned Dataset Overview and Export


| Step | Decision | Rows removed | Rationale |
|------|----------|-------------|-----------|
| Load `laliga_merged.csv` | Starting point from Section 4 | - | 3,600 player-season records |
| Filter MV ≥ €500k | Drop values below €500,000 | 292 | Sub-€500k entries are Transfermarkt placeholder values, not genuine market assessments; they form a separate low-quality mode in the log distribution |
| Filter time ≥ 450 min | Drop players below 450 minutes | varies | Per-90 metrics are statistically unreliable for very small samples; 450 min ≈ 5 full games is the standard analytics floor |
| Remove goalkeepers | Drop `position` starting with `GK` | varies | GK performance metrics (xG, xA, key passes, shots) are structurally incomparable with outfield players |
| Numeric conversion | Cast Understat numeric columns to float | 0 | API returns all fields as strings |
| Per-90 metrics | `xG_per90`, `xA_per90`, `goals_per90`, `assists_per90`, `npxG_per90`, `shots_per90` | 0 | Exposure-normalised metrics allow fair cross-player comparison regardless of minutes played |
| Log market value | `log_market_value = ln(market_value_in_eur)` | 0 | The distribution is heavily skewed; the log makes it more symmetric and suitable for regression |
| Position dummies | Midfielder and Forward columns (0/1) | 0 | Defender is the baseline comparison group |
| Season year | `season_year` (integer, e.g. 2014) | 0 | Enables time-trend control in regression |


In [32]:
pos_cols = []
for c in df.columns:
    if c.startswith('pos_'):
        pos_cols.append(c)
pos_cols = sorted(pos_cols)

# seleccionar y ordenar las columnas finales
columnas_finales = [
    # identificadores
    'player_id', 'player_name', 'season', 'season_year', 'team_title',
    # volumen
    'games', 'time',
    # estadisticas brutas
    'goals', 'assists', 'shots', 'key_passes', 'xG', 'xA', 'npg', 'npxG',
    # metricas per-90
    'xG_per90', 'xA_per90', 'goals_per90', 'assists_per90',
    'shots_per90', 'npxG_per90',
    # posicion
    'position', 'position_simple',
    # dummies de posicion (referencia: defender)
] + pos_cols + [
    # variable dependiente
    'market_value_in_eur', 'log_market_value',
    # metadato del merge
    'match_type',
]

# filtrar las que existen en el dataframe
final_cols = []
for c in columnas_finales:
    if c in df.columns:
        final_cols.append(c)
df_final = df[final_cols].copy()

print(f"Shape final: {df_final.shape[0]:,} filas × {df_final.shape[1]} columnas")
print(f"columnas: {list(df_final.columns)}")


Shape final: 2,448 filas × 28 columnas
columnas: ['player_id', 'player_name', 'season', 'season_year', 'team_title', 'games', 'time', 'goals', 'assists', 'shots', 'key_passes', 'xG', 'xA', 'npg', 'npxG', 'xG_per90', 'xA_per90', 'goals_per90', 'assists_per90', 'shots_per90', 'npxG_per90', 'position', 'position_simple', 'pos_Forward', 'pos_Midfielder', 'market_value_in_eur', 'log_market_value', 'match_type']


### 6.2 Summary statistics for key variables


In [33]:
# estadisticas descriptivas de variables continuas clave
key_vars = [
    'time', 'goals', 'assists', 'xG', 'xA', 'npxG',
    'xG_per90', 'xA_per90', 'goals_per90', 'assists_per90', 'npxG_per90',
    'market_value_in_eur', 'log_market_value',
]
df_final[key_vars].describe().round(4)


,time,goals,assists,xG,xA,npxG,xG_per90,xA_per90,goals_per90,assists_per90,npxG_per90,market_value_in_eur,log_market_value
count,2448.0000,2448.0000,2448.0000,2448.0000,2448.0000,2448.0000,2448.0000,2448.0000,2448.0000,2448.0000,2448.0000,2.448000e+03,2448.0000
mean,1758.9710,2.5633,1.7872,2.6790,1.8664,2.4458,0.1373,0.0954,0.1273,0.0901,0.1269,1.041424e+07,15.4105
std,755.6512,3.8375,2.2097,3.5170,1.9296,3.1130,0.1536,0.0836,0.1655,0.1021,0.1406,1.626787e+07,1.1736
min,450.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,5.000000e+05,13.1224
25%,1126.0000,0.0000,0.0000,0.5623,0.5119,0.5514,0.0320,0.0289,0.0000,0.0000,0.0317,2.000000e+06,14.5087
50%,1730.0000,1.0000,1.0000,1.3926,1.2136,1.3296,0.0767,0.0738,0.0694,0.0616,0.0736,4.000000e+06,15.2018
75%,2381.2500,3.0000,3.0000,3.2217,2.6102,2.9799,0.1908,0.1412,0.1783,0.1427,0.1707,1.000000e+07,16.1181
max,3413.0000,40.0000,16.0000,35.8337,14.7269,32.1177,1.4605,0.5588,1.1671,0.8327,1.4605,1.800000e+08,19.0085


In [34]:
# distribucion de temporadas y posiciones en el dataset final
print("distribucion por temporada:")
conteo = df_final.groupby('season_year').size().reset_index()
conteo.columns = ['season_year', 'n_records']
print(conteo.to_string(index=False))

print("\nPosition distribution (simplified):")
pos_dist = df_final['position_simple'].value_counts()
for pos, cnt in pos_dist.items():
    print(f"  {pos:<12} {cnt:,}  ({cnt/len(df_final)*100:.1f}%)")



distribucion por temporada:
 season_year  n_records
        2014        225
        2015        205
        2016        245
        2017        258
        2018        159
        2019        265
        2020        307
        2021        322
        2022        106
        2023        283
        2024         73

Position distribution (simplified):
  Defender     1,051  (42.9%)
  Forward      761  (31.1%)
  Midfielder   636  (26.0%)


In [35]:
# ultimo check de nulos
nulls = df_final.isnull().sum()
nulos = nulls[nulls > 0]

if len(nulos) == 0:
    print("sin valores faltantes en el dataset final")
else:
    print("valores faltantes:")
    pct = (nulos / len(df_final) * 100).round(2)
    print(pd.DataFrame({'missing': nulos, 'pct(%)': pct}).to_string())


sin valores faltantes en el dataset final


### 6.3 Save cleaned dataset


In [36]:
# guardar el dataset final
archivo_limpio = "laliga_cleaned.csv"
df_final.to_csv(archivo_limpio, index=False, encoding='utf-8')

import os
size_kb = os.path.getsize(archivo_limpio) / 1024
print(f"guardado:  {archivo_limpio}")
print(f"tamaño: {size_kb:.1f} KB")
print(f"filas y columnas: {df_final.shape[0]:,} rows × {df_final.shape[1]} columns")


guardado:  laliga_cleaned.csv
tamaño: 655.9 KB
filas y columnas: 2,448 rows × 28 columns


In [37]:
# vista previa del dataset limpio final
df_final.head(10)


,player_id,player_name,season,season_year,team_title,games,time,goals,assists,shots,...,assists_per90,shots_per90,npxG_per90,position,position_simple,pos_Forward,pos_Midfielder,market_value_in_eur,log_market_value,match_type
0,2270,Antoine Griezmann,2014/15,2014,Atletico Madrid,37,2490,22,1,78,...,0.036145,2.819277,0.531674,F M S,Forward,1,0,45000000,17.622173,exact
1,1125,Carlos Bacca,2014/15,2014,Sevilla,37,2581,20,6,69,...,0.209221,2.406044,0.464948,F S,Forward,1,0,25000000,17.034386,exact
2,2098,Luis Suárez,2014/15,2014,Barcelona,27,2182,16,14,75,...,0.577452,3.093492,0.599820,F S,Forward,1,0,80000000,18.197537,exact
3,2370,Karim Benzema,2014/15,2014,Real Madrid,29,2325,15,10,69,...,0.387097,2.670968,0.480481,F,Forward,1,0,50000000,17.727534,exact
4,4060,Sergio García,2014/15,2014,Espanyol,35,3030,14,9,80,...,0.267327,2.376238,0.260921,F M S,Forward,1,0,6000000,15.607270,exact
5,1732,Nolito,2014/15,2014,Celta Vigo,36,2970,13,13,105,...,0.393939,3.181818,0.244469,F M S,Forward,1,0,10000000,16.118096,exact
6,2251,Gareth Bale,2014/15,2014,Real Madrid,31,2585,13,9,103,...,0.313346,3.586074,0.465094,F M S,Forward,1,0,80000000,18.197537,exact
7,1709,Cristhian Stuani,2014/15,2014,Espanyol,37,1739,12,0,45,...,0.000000,2.328925,0.441061,F M S,Forward,1,0,4000000,15.201805,exact
8,2087,David Barral,2014/15,2014,Levante,35,2656,11,0,74,...,0.000000,2.507530,0.293864,F M S,Forward,1,0,2000000,14.508658,exact
9,2380,Paco Alcácer,2014/15,2014,Valencia,32,2179,11,5,61,...,0.206517,2.519504,0.427726,F S,Forward,1,0,20000000,16.811243,exact


---
## 7. Adding Player Age

Player age seems to be one of the most important factors in market value. Young players
with high performance are valued for their future potential, while older players depreciate
even if their current output remains strong.

I calculated age as the difference between `season_year` (the start of the football season)
and the player's birth year. I have also added `age_squared` to capture the non-linear
(inverted-U shaped) relationship between age and value.

**Data bridge:** `laliga_cleaned.csv` uses Understat player IDs. Age comes from
Transfermarkt's `players.csv`, which uses Transfermarkt player IDs.
I use `laliga_merged.csv` as a bridge to link both ID systems.


In [38]:
# cargar laliga_cleaned.csv y el puente de ids
df_clean = pd.read_csv("laliga_cleaned.csv")
df_bridge = pd.read_csv("laliga_merged.csv")[
    ['player_id', 'season', 'tm_player_id']
]
merged_check = df_clean.merge(
    df_bridge, on=['player_id', 'season'], how='inner'
)
sin_puente = df_clean.shape[0] - len(merged_check)
print(f"laliga_cleaned.csv: {df_clean.shape[0]:,} filas")
print(f"puente de ids: {len(df_bridge):,} filas")
print(f"registros sin puente: {sin_puente}")

laliga_cleaned.csv: 2,448 filas
puente de ids: 3,388 filas
registros sin puente: -39


In [39]:
# cargar date_of_birth desde players.csv
tm_dob = pd.read_csv("transfermarkt_data/players.csv")[['player_id', 'date_of_birth']]
tm_dob = tm_dob.rename(columns={'player_id': 'tm_player_id'})

# extraer año de nacimiento del string de fecha
tm_dob['birth_year'] = pd.to_datetime(tm_dob['date_of_birth'], errors='coerce').dt.year
print(f"Jugadores con fecha de nacimiento: {tm_dob['birth_year'].notna().sum():,} "
      f"de {len(tm_dob):,}")


Jugadores con fecha de nacimiento: 34,271 de 34,320


In [40]:
# calcular edad y unir al dataset limpio
bridge_raw = pd.read_csv("laliga_merged.csv")[
    ['player_id', 'season', 'tm_player_id', 'market_value_in_eur']
]
bridge_dob = bridge_raw.merge(
    tm_dob[['tm_player_id', 'birth_year']],
    on='tm_player_id', how='left'
)
# quedarse con una fila por jugador y temporada
bridge = (bridge_dob
          .sort_values('market_value_in_eur', ascending=False)
          .drop_duplicates(subset=['player_id', 'season'])
          .reset_index(drop=True))
bridge = bridge[['player_id', 'season', 'tm_player_id', 'birth_year']]
print(f"puente sin duplicados: {len(bridge)} filas")
df_with_dob = df_clean.merge(bridge, on=['player_id', 'season'], how='left')
df_with_dob['age']         = df_with_dob['season_year'] - df_with_dob['birth_year']
df_with_dob['age_squared'] = df_with_dob['age'] ** 2
n_con_edad = df_with_dob['age'].notna().sum()
n_sin_edad = df_with_dob['age'].isna().sum()
print(f"con edad calculada:          {n_con_edad}")
print(f"sin fecha de nacimiento:     {n_sin_edad}")


puente sin duplicados: 3338 filas
con edad calculada:          2447
sin fecha de nacimiento:     1


In [41]:
# filtrar edades fuera de rango logico
# rango razonable para jugadores de futbol profesional: 15–45 años
mask_age_ok = df_with_dob['age'].between(15, 45)
n_out_range = (~mask_age_ok & df_with_dob['age'].notna()).sum()

if n_out_range > 0:
    print(f"Edades fuera de rango (15–45): {n_out_range}")
    print(df_with_dob[~mask_age_ok & df_with_dob['age'].notna()][
        ['player_name', 'season', 'age']].to_string())
else:
    print("Todas las edades calculadas están en el rango 15–45. Sin anomalías.")

# distribucion de edad
print(f"\nDistribución de edad (jugadores con dato):")
print(df_with_dob['age'].describe().round(1).to_string())


Todas las edades calculadas están en el rango 15–45. Sin anomalías.

Distribución de edad (jugadores con dato):
count    2447.0
mean       26.8
std         4.0
min        16.0
25%        24.0
50%        27.0
75%        30.0
max        39.0


In [42]:
# construir dataset final con edad y guardar
# seleccionar columnas finales - incluir age y age_squared despues de season_year
cols_finales = [
    'player_id', 'player_name', 'season', 'season_year', 'team_title',
    'games', 'time',
    'goals', 'assists', 'shots', 'key_passes', 'xG', 'xA', 'npg', 'npxG',
    'xG_per90', 'xA_per90', 'goals_per90', 'assists_per90', 'shots_per90', 'npxG_per90',
    'position', 'position_simple', 'pos_Forward', 'pos_Midfielder',
    'age', 'age_squared',
    'market_value_in_eur', 'log_market_value',
    'match_type',
]

df_final = df_with_dob[cols_finales].copy()

print(f"Shape final (con edad): {df_final.shape}")
print(f"Nulos en 'age':         {df_final['age'].isna().sum():,}")
print(f"Nulos en 'age_squared': {df_final['age_squared'].isna().sum():,}")
print(f"\nPrimeras 5 filas con edad:")
print(df_final[['player_name', 'season', 'age', 'age_squared', 'market_value_in_eur']].head().to_string())

df_final.to_csv("laliga_cleaned.csv", index=False, encoding='utf-8')
print(f"\nArchivo actualizado guardado: laliga_cleaned.csv")


Shape final (con edad): (2448, 30)
Nulos en 'age':         1
Nulos en 'age_squared': 1

Primeras 5 filas con edad:
         player_name   season   age  age_squared  market_value_in_eur
0  Antoine Griezmann  2014/15  23.0        529.0             45000000
1       Carlos Bacca  2014/15  28.0        784.0             25000000
2        Luis Suárez  2014/15  27.0        729.0             80000000
3      Karim Benzema  2014/15  27.0        729.0             50000000
4      Sergio García  2014/15  31.0        961.0              6000000



Archivo actualizado guardado: laliga_cleaned.csv
